In [ ]:
import pandas as pd
import numpy as np

In [2]:
epiclem = pd.read_csv("C:/Users/Rachit/OneDrive/Documents/disease_survelliance/data/raw/Final_data.csv")
population = pd.read_csv("C:/Users/Rachit/OneDrive/Documents/disease_survelliance/data/raw/state_population.csv")

data1 = pd.DataFrame(epiclem)
data2 = pd.DataFrame(population)


In [3]:
# Mapping the epiclem states with population state data

state_mapping = {"Dadra and Nagar Haveli" : "Dadra and Nagar Haveli and Daman and Diu", 
                 "Daman and Diu": "Dadra and Nagar Haveli and Daman and Diu"}

data1['state_ut']=data1['state_ut'].replace(state_mapping)

print(data1['state_ut'].unique())

<StringArray>
[                               'Meghalaya',
                              'Maharashtra',
                               'Tamil Nadu',
                                  'Gujarat',
                                   'Kerala',
                            'Uttar Pradesh',
                                   'Odisha',
                              'West Bengal',
                                'Karnataka',
                                'Telangana',
                                  'Haryana',
                           'Andhra Pradesh',
                           'Madhya Pradesh',
                                   'Punjab',
                                'Jharkhand',
                             'Chhattisgarh',
                                'Rajasthan',
                                    'Assam',
                        'Arunachal Pradesh',
                               'Puducherry',
                              'Uttarakhand',
                         'Himachal Prades

In [4]:
data1['Disease'].unique()

<StringArray>
[        'Acute Diarrhoeal Disease',                          'Malaria',
      'Acute Encephalitis Syndrome',            'Acute Gastroenteritis',
                           'Dengue',        'pyrexia of unknown origin',
                      'Chikungunya',                          'Cholera',
                     'Malaria (PV)',                     'Dengue Fever',
                 'Suspected Dengue',               'Dengue Chikungunya',
           'Dengue And Chikungunya',                'Suspected Cholera',
                         'Diarrhea',            'Suspected Chikungunya',
 'Suspected Dengue And Chikungunya',                  'Gastroenteritis',
               'Dengue And Malaria',               'Dengue/Chikungunya',
               'Chikungunya/Dengue',              'Chikungunya/ Dengue']
Length: 22, dtype: str

In [5]:
# Mapping the single and combination of diseases into groups

disease_mapping ={
# 
    'Suspected Dengue': 'Dengue',
    'Diarrhea': 'Acute Diarrhoeal Disease',
    'Suspected Chikungunya': 'Chikungunya',
    'Suspected Cholera': 'Cholera',
    'Dengue Fever': 'Dengue',
    'Gastroenteritis': 'Acute Gastroenteritis',
    'Malaria (PV)': 'Malaria',

# co-infections
    'Suspected Dengue And Chikungunya': 'Dengue And Chikungunya',
    'Chikungunya/Dengue' : 'Dengue And Chikungunya',
    'Chikungunya/ Dengue': 'Dengue And Chikungunya',
    'Dengue/Chikungunya':  'Dengue And Chikungunya',
    'Dengue Chikungunya':  'Dengue And Chikungunya'

}

data1['Disease_Grouped']= data1['Disease'].replace(disease_mapping)
print(data1['Disease_Grouped'].unique())

<StringArray>
[   'Acute Diarrhoeal Disease',                     'Malaria',
 'Acute Encephalitis Syndrome',       'Acute Gastroenteritis',
                      'Dengue',   'pyrexia of unknown origin',
                 'Chikungunya',                     'Cholera',
      'Dengue And Chikungunya',          'Dengue And Malaria']
Length: 10, dtype: str


In [6]:
# Handling missing values of LAI, preci, Temp columns

for col in ['LAI','Temp','preci']:
    data1[col]=data1.groupby(['state_ut','mon'])[col].transform(
        lambda x: x.fillna(x.median())
    )

for col in ['preci', 'LAI', 'Temp']:
    data1[col]=data1[col].fillna(data1[col].median())

for col in ['preci', 'LAI', 'Temp']:
    print(data1[col].isnull().sum())

0
0
0


In [7]:
#Handling missing deaths - Since 71% of data is missing, we can neither drop them nor fill it with aggregated value.
# We are just filling the empty rows with 0 for only aggregation purpose

# Creating a copy of Death column
data1['epiclem_deaths']= data1['Deaths'].copy()

# Tracking original missing rows
data1['Available_Deaths']= data1['Deaths'].notna()

# Filling the null values with 0
data1['Deaths']=data1['Deaths'].fillna(0)

In [8]:
def season(month):
    if month in [12,1,2]:
        return 'Winter'
    if month in [3,4,5]:
        return 'Summer'
    if month in [6,7,8,9]:
        return 'Monsoon'
    else:
        return 'Post Monsoon'

data1['Season']= data1['mon'].apply(season)
    

In [9]:
data1['Cases'] = pd.to_numeric(data1['Cases'], errors='coerce')
data1 = data1.dropna(subset=['Cases'])
data1['Cases'] = data1['Cases'].astype(int)

print(data1['Cases'].dtype)

int64


In [12]:
data1=data1.merge(data2,left_on='state_ut', right_on='State',how='left')

print(data1.columns)

Index(['Unnamed: 0', 'week_of_outbreak', 'state_ut', 'district', 'Disease',
       'Cases', 'Deaths', 'day', 'mon', 'year', 'Latitude', 'Longitude',
       'preci', 'LAI', 'Temp', 'Disease_Grouped', 'epiclem_deaths',
       'Available_Deaths', 'Season', 'State', 'Population_2024'],
      dtype='str')


In [13]:
#Calculating Cases_Per_Lakh metric

data1['Cases_Per_Lakh'] = (data1['Cases']/data1['Population_2024'])*100000
data1['Cases_Per_Lakh'] = data1['Cases_Per_Lakh'].round(4)

In [14]:
data1.columns
data1.dtypes

Unnamed: 0            int64
week_of_outbreak        str
state_ut                str
district                str
Disease                 str
Cases                 int64
Deaths              float64
day                   int64
mon                   int64
year                  int64
Latitude            float64
Longitude           float64
preci               float64
LAI                 float64
Temp                float64
Disease_Grouped         str
epiclem_deaths      float64
Available_Deaths       bool
Season                  str
State                   str
Population_2024       int64
Cases_Per_Lakh      float64
dtype: object

In [15]:
# Exporting processed data

#Drop unnecessary columns
cleaned_data= data1.drop(columns=['State','Unnamed: 0'])

cleaned_data.to_csv("C:/Users/Rachit/OneDrive/Documents/disease_survelliance/data/processed/epiclem.csv", index=False)